In [21]:
import torch
print(f"GPU 사용 가능 여부: {torch.cuda.is_available()}")
print(f"현재 GPU 이름: {torch.cuda.get_device_name(0)}")

GPU 사용 가능 여부: True
현재 GPU 이름: Tesla T4


In [66]:
!pip install python_speech_features soundfile

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import soundfile as sf
import python_speech_features
from pathlib import Path
import random

In [67]:
import os
import shutil
import glob

# 새 바구니 만들기
target_root = "/content/syncnet_dataset/syncnet_data"
os.makedirs(target_root, exist_ok=True)

names = ['준형', '사랑', '초원', '지원']

for name in names:
    zip_path = f'/content/drive/MyDrive/SyncNet_Project/data/syncnet_data_{name}.zip'
    print(f"📦 {name} 님 데이터 압축 해제 및 정렬 중...")

    # 임시 폴더에 풀기
    !unzip -qo "{zip_path}" -d /content/temp_extract

    # 1. 맥용 쓰레기 파일(._)은 아예 옮기기 전에 미리 삭제 (원천 차단)
    !find /content/temp_extract -name "._*" -delete

    # 2. 진짜 'positive', 'negative' 폴더 내용물만 타겟으로 이동
    for label in ['positive', 'negative']:
        # 어떤 깊이에 있든 positive/negative 폴더를 찾음
        found_folders = glob.glob(f'/content/temp_extract/**/{label}', recursive=True)
        for src in found_folders:
            dest = os.path.join(target_root, label)
            os.makedirs(dest, exist_ok=True)
            for item in os.listdir(src):
                s_item = os.path.join(src, item)
                d_item = os.path.join(dest, item)
                if not os.path.exists(d_item): # 중복 아니면 이동
                    shutil.move(s_item, d_item)

    # 임시 폴더 비우기
    !rm -rf /content/temp_extract

# 최종 확인
p_count = len(os.listdir(os.path.join(target_root, 'positive')))
n_count = len(os.listdir(os.path.join(target_root, 'negative')))
print(f"\n✨ 완료! 진짜 데이터만 모였습니다. (Positive: {p_count}, Negative: {n_count})")

📦 준형 님 데이터 압축 해제 및 정렬 중...
📦 사랑 님 데이터 압축 해제 및 정렬 중...
📦 초원 님 데이터 압축 해제 및 정렬 중...
📦 지원 님 데이터 압축 해제 및 정렬 중...

✨ 완료! 진짜 데이터만 모였습니다. (Positive: 41, Negative: 43)


In [68]:
import os
import shutil

# 데이터 경로 설정
pos_path = '/content/syncnet_dataset/syncnet_data/positive'
neg_path = '/content/syncnet_dataset/syncnet_data/negative'

# 현재 폴더 리스트 확보
pos_list = set(os.listdir(pos_path))
neg_list = set(os.listdir(neg_path))

# 1. 서로 다른 데이터 찾아내기
only_in_pos = pos_list - neg_list
only_in_neg = neg_list - pos_list

print(f"🔍 불일치 분석 결과:")
print(f"- Positive에만 있는 데이터 (Negative 없음): {only_in_pos}")
print(f"- Negative에만 있는 데이터 (Positive 없음): {only_in_neg}")

# 2. 짝이 없는 폴더 삭제 (데이터 정제)
print("\n🧹 짝 없는 데이터를 삭제하여 균형을 맞춥니다...")

for folder in only_in_pos:
    shutil.rmtree(os.path.join(pos_path, folder))
    print(f"🗑️ 삭제됨 (Positive 전용): {folder}")

for folder in only_in_neg:
    shutil.rmtree(os.path.join(neg_path, folder))
    print(f"🗑️ 삭제됨 (Negative 전용): {folder}")

# 3. 최종 결과 확인
final_pos = os.listdir(pos_path)
final_neg = os.listdir(neg_path)

print("\n✨ 정제 완료!")
print(f"최종 Positive 개수: {len(final_pos)}")
print(f"최종 Negative 개수: {len(final_neg)}")
print(f"이제 모든 데이터가 {len(final_pos)}쌍으로 완벽하게 일치합니다.")

🔍 불일치 분석 결과:
- Positive에만 있는 데이터 (Negative 없음): set()
- Negative에만 있는 데이터 (Positive 없음): {'20260219_220628', '20260219_215003'}

🧹 짝 없는 데이터를 삭제하여 균형을 맞춥니다...
🗑️ 삭제됨 (Negative 전용): 20260219_220628
🗑️ 삭제됨 (Negative 전용): 20260219_215003

✨ 정제 완료!
최종 Positive 개수: 41
최종 Negative 개수: 41
이제 모든 데이터가 41쌍으로 완벽하게 일치합니다.


In [69]:
import sys
SYNCNET_PATH = "/content/drive/MyDrive/SyncNet_Project/video-detect/syncnet_python"
if SYNCNET_PATH not in sys.path:
    sys.path.insert(0, SYNCNET_PATH)

from SyncNetInstance import SyncNetInstance
print("SyncNet 모듈 로드 성공!")

SyncNet 모듈 로드 성공!


In [70]:
class SyncNetDataset(Dataset):
    def __init__(self, root_dir, video_fps=25, audio_sr=16000):
        self.root_dir = root_dir
        self.samples = []

        target_root = os.path.join(root_dir, "syncnet_data")
        pos_root = os.path.join(target_root, 'positive')
        neg_root = os.path.join(target_root, 'negative')

        def is_valid(name):
            # '._'로 시작하는 숨김 파일은 무시함
            return not name.startswith('._')

        if os.path.exists(pos_root):
            # 1. Positive 로드
            for sid in sorted(os.listdir(pos_root)):
                if not is_valid(sid): continue # 숨김 폴더 건너뛰기

                p_path = os.path.join(pos_root, sid)
                v_p, a_p = os.path.join(p_path, 'video.mp4'), os.path.join(p_path, 'audio.wav')

                if os.path.exists(v_p) and os.path.exists(a_p):
                    self.samples.append({'video': v_p, 'audio': a_p, 'label': 1})

            # 2. Negative 로드
            for sid in sorted(os.listdir(neg_root)):
                if not is_valid(sid): continue

                n_path = os.path.join(neg_root, sid)
                if os.path.isdir(n_path):
                    for sub in os.listdir(n_path):
                        if not is_valid(sub): continue

                        sub_path = os.path.join(n_path, sub)
                        v_n, a_n = os.path.join(sub_path, 'video.mp4'), os.path.join(sub_path, 'audio.wav')
                        if os.path.exists(v_n) and os.path.exists(a_n):
                            self.samples.append({'video': v_n, 'audio': a_n, 'label': -1})

        print(f"✅ 필터링 완료! {len(self.samples)}개의 유효 샘플 확보.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        # 1. 비디오 전처리 (안전 장치 추가)
        cap = cv2.VideoCapture(s['video'])
        frames = []
        for _ in range(5): # 정확히 5프레임 추출 시도
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)
        cap.release()

        # 프레임이 부족할 경우 마지막 프레임 복사 (Padding)
        while len(frames) < 5:
            if len(frames) == 0: # 아예 로드 실패 시 검은 화면 생성
                frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
            else:
                frames.append(frames[-1])

        v = np.array(frames).astype(np.float32) # (T, H, W, C)
        # 차원 변환 오류 방지를 위해 명시적 확인
        try:
            v = torch.from_numpy(np.transpose(v, (3, 0, 1, 2))) # (C, T, H, W)
        except ValueError as e:
            print(f"Error at {s['video']}: {e}")
            v = torch.zeros((3, 5, 224, 224))

        # 2. 오디오 전처리
        audio, _ = sf.read(s['audio'])
        mfcc = python_speech_features.mfcc(audio, 16000).T.astype(np.float32)
        if mfcc.shape[1] > 20:
            mfcc = mfcc[:, :20]
        else:
            mfcc = np.pad(mfcc, ((0,0), (0, 20-mfcc.shape[1])))
        a = torch.from_numpy(mfcc).unsqueeze(0) # (1, C, T)

        return v, a, torch.tensor(s['label'], dtype=torch.float32)

In [71]:
import os
model_path = "/content/drive/MyDrive/SyncNet_Project/video-detect/syncnet_python/data/syncnet_v2.model"


if os.path.exists(model_path):
    size = os.path.getsize(model_path) / (1024 * 1024) # MB 단위 계산
    print(f"현재 파일 크기: {size:.2f} MB")
    if size > 10: # 52MB라면 이 조건을 통과합니다.
        print("파일 준비되었습니다. 학습을 시작해도 좋습니다.")
    else:
        print("파일이 너무 작습니다. 업로드 과정을 다시 확인해주세요.")
else:
    print("파일이 해당 경로에 보이지 않습니다.")

현재 파일 크기: 52.04 MB
파일 준비되었습니다. 학습을 시작해도 좋습니다.


In [72]:
from SyncNetInstance import SyncNetInstance

# 1. 모델 초기화 및 가중치 로드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = SyncNetInstance(dropout=0, num_layers_in_fc_layers=1024).to(device)

model_path = "/content/drive/MyDrive/SyncNet_Project/video-detect/syncnet_python/data/syncnet_v2.model"

try:
    # 수동으로 올린 52MB 파일을 로드합니다.
    net.loadParameters(model_path)
    print("✅ 프리트레인 모델 가중치 로드 성공!")
except Exception as e:
    print(f"모델 로드 실패: {e}")

✅ 프리트레인 모델 가중치 로드 성공!


In [73]:
# 데이터셋 인스턴스 생성
dataset = SyncNetDataset(root_dir='/content/syncnet_dataset')

# 데이터로더 설정 (배치 사이즈는 T4 GPU 메모리에 맞게 8로 설정)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

print(f"✅ 학습 준비 완료! 총 {len(dataset)}개의 샘플을 사용합니다.")

✅ 필터링 완료! 240개의 유효 샘플 확보.
✅ 학습 준비 완료! 총 240개의 샘플을 사용합니다.


In [74]:
# 1. 맥용 숨김 메타데이터 파일 (._ 시작) 삭제
!find /content/syncnet_dataset -name "._*" -delete

# 2. 혹시 모를 __MACOSX 폴더도 삭제
!rm -rf /content/syncnet_dataset/__MACOSX

In [82]:
# 사용하는 Loss: Contrastive Loss
# Positive Loss (정답 쌍): 비디오와 오디오 사이 유클리드 거리를 0에 가깝게 줄이도록 학습
# Negative Loss (오답 쌍): 일치하지 않는 쌍에 대해서는 거리가 최소한 설정된 마진 이상이 되도록 멀어지게 만듬

In [75]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

# 1. 모델 초기화 (기존 가중치로 깨끗하게 시작)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = SyncNetInstance(dropout=0, num_layers_in_fc_layers=1024).to(device)

# 가중치 로드 경로 확인 필요
model_path = "/content/drive/MyDrive/SyncNet_Project/video-detect/syncnet_python/data/syncnet_v2.model"
net.loadParameters(model_path)

# 2. 하이퍼파라미터 및 설정
# 1e-5로 소폭 상향하여 학습의 '동력'을 줍니다.
optimizer = optim.Adam(net.parameters(), lr=1e-5)
margin = 1.0
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

net.train()
print("🚀 [최종 개선 모드] In-batch Shuffling 기반 파인튜닝 시작")

for epoch in range(20): # 데이터가 적으므로 20회 정도 충분히 학습
    epoch_loss = 0
    for v, a, label in dataloader:
        v, a, label = v.to(device), a.to(device), label.to(device)

        optimizer.zero_grad()

        # 특징 추출 및 정규화
        v_feat = F.normalize(net.__S__.forward_lip(v), p=2, dim=1)
        a_feat = F.normalize(net.__S__.forward_aud(a), p=2, dim=1)

        # --- 핵심: Contrastive Loss 직접 계산 ---

        # 1. Positive Loss: 정답 쌍(Label 1)의 거리는 0에 가깝게
        dist_pos = F.pairwise_distance(v_feat, a_feat)
        # 실제 Positive 데이터만 필터링 (데이터셋 구성에 따라 label 활용)
        loss_pos = torch.pow(dist_pos[label == 1], 2).mean() if (label == 1).any() else 0

        # 2. Negative Loss (강력한 자극): 배치의 오디오를 한 칸씩 밀어서 오답 쌍 생성
        # 이렇게 하면 190개 데이터가 훨씬 많은 조합으로 늘어납니다.
        a_feat_neg = torch.roll(a_feat, shifts=1, dims=0)
        dist_neg = F.pairwise_distance(v_feat, a_feat_neg)

        # 오답 쌍의 거리는 최소 margin(1.0) 이상이 되도록 강제
        loss_neg = torch.pow(torch.clamp(margin - dist_neg, min=0.0), 2).mean()

        # 전체 손실 합산
        loss = loss_pos + loss_neg

        if loss != 0:
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

    # 2에포크마다 진행 상황 출력
    if (epoch + 1) % 2 == 0:
        print(f"에포크 [{epoch+1}/20] - Avg Loss: {epoch_loss/len(dataloader):.6f}")

# 3. 모델 저장
save_path = "/content/drive/MyDrive/SyncNet_Project/syncnet_finetuned_final.pth"
torch.save(net.state_dict(), save_path)
print(f"✅ 학습 완료! 모델이 저장되었습니다: {save_path}")

🚀 [최종 개선 모드] In-batch Shuffling 기반 파인튜닝 시작
에포크 [2/20] - Avg Loss: 1.292534
에포크 [4/20] - Avg Loss: 1.160180
에포크 [6/20] - Avg Loss: 0.896050
에포크 [8/20] - Avg Loss: 0.651169
에포크 [10/20] - Avg Loss: 0.494102
에포크 [12/20] - Avg Loss: 0.415865
에포크 [14/20] - Avg Loss: 0.464031
에포크 [16/20] - Avg Loss: 0.400863
에포크 [18/20] - Avg Loss: 0.295448
에포크 [20/20] - Avg Loss: 0.258461
✅ 학습 완료! 모델이 저장되었습니다: /content/drive/MyDrive/SyncNet_Project/syncnet_finetuned_final.pth


In [83]:
# 비교 지표: Confidence Score
# 파인튜닝 전 (v2.model) vs 파인튜닝 후 (pth)
# Positive Score -> 파인튜닝 후 상승: 맞는 싱크를 더 확신하게 됨
# Negative Score -> 파인튜닝 후 하락: 안 맞는 싱크(Unsync)를 더 잘 걸러냄

In [79]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import python_speech_features
import soundfile as sf
import os

def calculate_sync_metrics(video_path, audio_path, model, device):
    """비디오와 오디오의 유사도(Confidence)와 거리(Distance)를 계산합니다."""
    # 1. 비디오/오디오 전처리 (기존 로직 유지하되 안전하게 처리)
    cap = cv2.VideoCapture(video_path)
    frames = []
    for _ in range(5):
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (224, 224))
        frames.append(frame)
    cap.release()

    if len(frames) < 5: return 0, 10 # 실패 시 안전값 반환

    v = torch.from_numpy(np.transpose(np.array(frames).astype(np.float32), (3, 0, 1, 2))).unsqueeze(0).to(device)

    audio, _ = sf.read(audio_path)
    mfcc = python_speech_features.mfcc(audio, 16000).T.astype(np.float32)
    if mfcc.shape[1] > 20: mfcc = mfcc[:, :20]
    else: mfcc = np.pad(mfcc, ((0,0), (0, 20-mfcc.shape[1])))
    a = torch.from_numpy(mfcc).unsqueeze(0).unsqueeze(0).to(device)

    # 2. 특징 추출
    model.eval()
    with torch.no_grad():
        v_feat = F.normalize(model.__S__.forward_lip(v), p=2, dim=1)
        a_feat = F.normalize(model.__S__.forward_aud(a), p=2, dim=1)

        # 유클리드 거리 계산
        dist = F.pairwise_distance(v_feat, a_feat).item()
        # 점수화: 거리가 가까울수록(0) 점수가 높고(10), 멀수록(2) 점수가 낮음(0)
        conf = max(0, 10 * (1 - dist/2))

    return conf, dist



In [88]:
# --- 실행 부분 ---
device = torch.device("cuda")

# 1. 모델 로드
net_old = SyncNetInstance(dropout=0, num_layers_in_fc_layers=1024).to(device)
net_old.loadParameters("/content/drive/MyDrive/SyncNet_Project/video-detect/syncnet_python/data/syncnet_v2.model")

net_new = SyncNetInstance(dropout=0, num_layers_in_fc_layers=1024).to(device)
net_new.load_state_dict(torch.load("/content/drive/MyDrive/SyncNet_Project/syncnet_finetuned_final.pth"))

# 2. 테스트 데이터 설정 (Positive 하나, Negative 하나 준비하세요)
pos_v = "/content/syncnet_dataset/syncnet_data/positive/20260218_111411/video.mp4"
pos_a = "/content/syncnet_dataset/syncnet_data/positive/20260218_111411/audio.wav"

neg_v = "/content/syncnet_dataset/syncnet_data/negative/20260218_111411/neg_04_+447ms/video.mp4"
neg_a = "/content/syncnet_dataset/syncnet_data/negative/20260218_111411/neg_04_+447ms/audio.wav"

test_cases = [("✅ Positive (정답)", pos_v, pos_a), ("❌ Negative (오답)", neg_v, neg_a)]

print(f"{'테스트 타입':<20} | {'기존 모델 점수':<15} | {'파인튜닝 점수':<15} | {'변화량'}")
print("-" * 75)

for title, v_p, a_p in test_cases:
    if not os.path.exists(v_p):
        print(f"파일 없음: {v_p}")
        continue

    s_old, d_old = calculate_sync_metrics(v_p, a_p, net_old, device)
    s_new, d_new = calculate_sync_metrics(v_p, a_p, net_new, device)

    diff = s_new - s_old
    print(f"{title:<18} | {s_old:>12.4f} | {s_new:>12.4f} | {diff:>+8.4f}")

print("-" * 75)
print("💡 팁: Positive 점수는 올라가고, Negative 점수는 낮아졌다면 완벽한 파인튜닝입니다!")

테스트 타입               | 기존 모델 점수        | 파인튜닝 점수         | 변화량
---------------------------------------------------------------------------
✅ Positive (정답)    |       3.8608 |       6.6661 |  +2.8052
❌ Negative (오답)    |       7.6359 |       1.8042 |  -5.8317
---------------------------------------------------------------------------
💡 팁: Positive 점수는 올라가고, Negative 점수는 낮아졌다면 완벽한 파인튜닝입니다!


In [87]:
# --- 실행 부분 ---
device = torch.device("cuda")

# 1. 모델 로드
net_old = SyncNetInstance(dropout=0, num_layers_in_fc_layers=1024).to(device)
net_old.loadParameters("/content/drive/MyDrive/SyncNet_Project/video-detect/syncnet_python/data/syncnet_v2.model")

net_new = SyncNetInstance(dropout=0, num_layers_in_fc_layers=1024).to(device)
net_new.load_state_dict(torch.load("/content/drive/MyDrive/SyncNet_Project/syncnet_finetuned_final.pth"))

# 2. 테스트 데이터 설정 (Positive 하나, Negative 하나 준비하세요)
pos_v = "/content/syncnet_dataset/syncnet_data/positive/20260218_200837/video.mp4"
pos_a = "/content/syncnet_dataset/syncnet_data/positive/20260218_200837/audio.wav"

neg_v = "/content/syncnet_dataset/syncnet_data/negative/20260218_200837/neg_02_-732ms/video.mp4"
neg_a = "/content/syncnet_dataset/syncnet_data/negative/20260218_200837/neg_02_-732ms/audio.wav"

test_cases = [("✅ Positive (정답)", pos_v, pos_a), ("❌ Negative (오답)", neg_v, neg_a)]

print(f"{'테스트 타입':<20} | {'기존 모델 점수':<15} | {'파인튜닝 점수':<15} | {'변화량'}")
print("-" * 75)

for title, v_p, a_p in test_cases:
    if not os.path.exists(v_p):
        print(f"파일 없음: {v_p}")
        continue

    s_old, d_old = calculate_sync_metrics(v_p, a_p, net_old, device)
    s_new, d_new = calculate_sync_metrics(v_p, a_p, net_new, device)

    diff = s_new - s_old
    print(f"{title:<18} | {s_old:>12.4f} | {s_new:>12.4f} | {diff:>+8.4f}")

print("-" * 75)
print("💡 팁: Positive 점수는 올라가고, Negative 점수는 낮아졌다면 완벽한 파인튜닝입니다!")

테스트 타입               | 기존 모델 점수        | 파인튜닝 점수         | 변화량
---------------------------------------------------------------------------
✅ Positive (정답)    |       6.2879 |       8.6709 |  +2.3830
❌ Negative (오답)    |       5.9350 |       3.2190 |  -2.7161
---------------------------------------------------------------------------
💡 팁: Positive 점수는 올라가고, Negative 점수는 낮아졌다면 완벽한 파인튜닝입니다!


In [86]:
# --- 실행 부분 ---
device = torch.device("cuda")

# 1. 모델 로드
net_old = SyncNetInstance(dropout=0, num_layers_in_fc_layers=1024).to(device)
net_old.loadParameters("/content/drive/MyDrive/SyncNet_Project/video-detect/syncnet_python/data/syncnet_v2.model")

net_new = SyncNetInstance(dropout=0, num_layers_in_fc_layers=1024).to(device)
net_new.load_state_dict(torch.load("/content/drive/MyDrive/SyncNet_Project/syncnet_finetuned_final.pth"))

# 2. 테스트 데이터 설정 (Positive 하나, Negative 하나 준비하세요)
pos_v = "/content/syncnet_dataset/syncnet_data/positive/20260218_150346/video.mp4"
pos_a = "/content/syncnet_dataset/syncnet_data/positive/20260218_150346/audio.wav"

neg_v = "/content/syncnet_dataset/syncnet_data/negative/20260218_150346/neg_00_+787ms/video.mp4"
neg_a = "/content/syncnet_dataset/syncnet_data/negative/20260218_150346/neg_00_+787ms/audio.wav"

test_cases = [("✅ Positive (정답)", pos_v, pos_a), ("❌ Negative (오답)", neg_v, neg_a)]

print(f"{'테스트 타입':<20} | {'기존 모델 점수':<15} | {'파인튜닝 점수':<15} | {'변화량'}")
print("-" * 75)

for title, v_p, a_p in test_cases:
    if not os.path.exists(v_p):
        print(f"파일 없음: {v_p}")
        continue

    s_old, d_old = calculate_sync_metrics(v_p, a_p, net_old, device)
    s_new, d_new = calculate_sync_metrics(v_p, a_p, net_new, device)

    diff = s_new - s_old
    print(f"{title:<18} | {s_old:>12.4f} | {s_new:>12.4f} | {diff:>+8.4f}")

print("-" * 75)
print("💡 팁: Positive 점수는 올라가고, Negative 점수는 낮아졌다면 완벽한 파인튜닝입니다!")

테스트 타입               | 기존 모델 점수        | 파인튜닝 점수         | 변화량
---------------------------------------------------------------------------
✅ Positive (정답)    |       5.6092 |       8.0213 |  +2.4122
❌ Negative (오답)    |       6.6880 |       3.7981 |  -2.8899
---------------------------------------------------------------------------
💡 팁: Positive 점수는 올라가고, Negative 점수는 낮아졌다면 완벽한 파인튜닝입니다!


In [85]:
# --- 실행 부분 ---
device = torch.device("cuda")

# 1. 모델 로드
net_old = SyncNetInstance(dropout=0, num_layers_in_fc_layers=1024).to(device)
net_old.loadParameters("/content/drive/MyDrive/SyncNet_Project/video-detect/syncnet_python/data/syncnet_v2.model")

net_new = SyncNetInstance(dropout=0, num_layers_in_fc_layers=1024).to(device)

net_new.load_state_dict(torch.load("/content/drive/MyDrive/SyncNet_Project/syncnet_finetuned_final.pth"))

# 2. 테스트 데이터 설정 (Positive 하나, Negative 하나 준비하세요)
pos_v = "/content/syncnet_dataset/syncnet_data/positive/20260219_220749/video.mp4"
pos_a = "/content/syncnet_dataset/syncnet_data/positive/20260219_220749/audio.wav"

neg_v = "/content/syncnet_dataset/syncnet_data/negative/20260219_220749/neg_03_+538ms/video.mp4"
neg_a = "/content/syncnet_dataset/syncnet_data/negative/20260219_220749/neg_03_+538ms/audio.wav"

test_cases = [("✅ Positive (정답)", pos_v, pos_a), ("❌ Negative (오답)", neg_v, neg_a)]

print(f"{'테스트 타입':<20} | {'기존 모델 점수':<15} | {'파인튜닝 점수':<15} | {'변화량'}")
print("-" * 75)

for title, v_p, a_p in test_cases:
    if not os.path.exists(v_p):
        print(f"파일 없음: {v_p}")
        continue

    s_old, d_old = calculate_sync_metrics(v_p, a_p, net_old, device)
    s_new, d_new = calculate_sync_metrics(v_p, a_p, net_new, device)

    diff = s_new - s_old
    print(f"{title:<18} | {s_old:>12.4f} | {s_new:>12.4f} | {diff:>+8.4f}")

print("-" * 75)
print("💡 팁: Positive 점수는 올라가고, Negative 점수는 낮아졌다면 완벽한 파인튜닝입니다!")

테스트 타입               | 기존 모델 점수        | 파인튜닝 점수         | 변화량
---------------------------------------------------------------------------
✅ Positive (정답)    |       2.4546 |       8.1326 |  +5.6779
❌ Negative (오답)    |       3.4272 |       3.1722 |  -0.2550
---------------------------------------------------------------------------
💡 팁: Positive 점수는 올라가고, Negative 점수는 낮아졌다면 완벽한 파인튜닝입니다!
